# Hybrid Model Training with Validation

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
# LOADING DATASET
df = pd.read_csv('Train_Cropyield_Model.csv')

In [3]:
df.head()

,Unnamed: 0,State,District,longitude,latitude,Average_avg-Temp,Average-Min Temp,Average-max-temp,avg-precipitation,avg-windSpeed,PH,Clay,Sand,Silt,Crop Yield,Hectare,Soil_Total
0,0.0,Abia,Abayi Abia,7.335453,5.127194,26.444297,22.225578,30.664500,194.166667,1.503953,5.400000,25.166667,66.166667,8.833333,58.322681,41.39817,100.166667
1,1.0,Abia,Abayi Abia,7.335453,5.127194,26.600547,22.346411,30.860333,197.916667,1.554130,5.583333,25.666667,51.833333,15.333333,58.322681,41.39817,92.833333
2,2.0,Abia,Abayi Abia,7.335453,5.127194,26.504068,22.043109,30.867396,158.760417,1.550376,5.800000,22.600000,60.000000,17.000000,58.322681,41.39817,99.600000
3,3.0,Abia,Abiriba Abia,7.730807,5.709564,26.458135,22.252682,30.667526,175.625000,1.503953,5.400000,25.166667,66.166667,8.833333,58.322681,41.39817,100.166667
4,4.0,Abia,Achara-Uturu Abia,7.431694,5.840898,26.479510,22.243865,30.709562,171.583333,1.474526,5.833333,22.166667,66.500000,10.000000,58.322681,41.39817,98.666667


In [ ]:
# Dropped useless index column
df.drop(columns=["Unnamed: 0"], inplace=True, errors='ignore')

In [5]:
df.head(1)

,State,District,longitude,latitude,Average_avg-Temp,Average-Min Temp,Average-max-temp,avg-precipitation,avg-windSpeed,PH,Clay,Sand,Silt,Crop Yield,Hectare,Soil_Total
0,Abia,Abayi Abia,7.335453,5.127194,26.444297,22.225578,30.6645,194.166667,1.503953,5.4,25.166667,66.166667,8.833333,58.322681,41.39817,100.166667


In [8]:
print(df["avg-precipitation"].unique().max())
print(df["avg-precipitation"].unique().min())
print(df["avg-precipitation"].unique().mean())


314.0568794
5.606800199999995
155.20006751101013


In [9]:
print(df["avg-windSpeed"].unique().max())
print(df["avg-windSpeed"].unique().min())
print(df["avg-windSpeed"].unique().mean())

2.6646848645000003
0.6644140604999998
1.660368019103998


In [ ]:
# DEFINING FEATURE GROUPS
rf_features = ['longitude', 'latitude', 'Average_avg-Temp', 'Average-Min Temp', 
               'Average-max-temp', 'avg-precipitation', 'avg-windSpeed', 'PH', 'Clay', 'Sand', 'Silt']

lr_features = ['State', 'District', 'Hectare', 'longitude', 'latitude']

In [ ]:
# PREPARING CATEGORICAL DATA FOR LINEAR REGRESSION
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoded_cats = encoder.fit_transform(df[['State', 'District']])
encoded_cat_names = encoder.get_feature_names_out(['State', 'District'])

In [22]:
# Combine encoded names with numerical columns for LR data
X_lr_full = pd.concat([
    pd.DataFrame(encoded_cats, columns=encoded_cat_names),
    df[['Hectare', 'longitude', 'latitude']].reset_index(drop=True)
], axis=1)

X_rf_full = df[rf_features]
y = df['Crop Yield']

In [ ]:
# SPLIT DATA (Train: 70%, Validation: 15%, Test: 15%)
idx_train, idx_temp = train_test_split(df.index, test_size=0.3, random_state=42)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.5, random_state=42)

# Create sets for Training
X_rf_train, y_train = X_rf_full.loc[idx_train], y.loc[idx_train]
X_lr_train = X_lr_full.loc[idx_train]

# Create sets for Validation
X_rf_val, y_val = X_rf_full.loc[idx_val], y.loc[idx_val]
X_lr_val = X_lr_full.loc[idx_val]

# Create sets for Testing
X_rf_test, y_test = X_rf_full.loc[idx_test], y.loc[idx_test]
X_lr_test = X_lr_full.loc[idx_test]

In [ ]:
# Creating Helper function for metrics
def evaluate_model(name, y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"--- {name} ---")
    print(f"R-squared: {r2:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}\n")


In [ ]:
# TRAIN AND EVALUATE INDIVIDUAL MODELS (VALIDATION FIRST)

print("### VALIDATION SET RESULTS ###\n")

# Model 1: Random Forest (Validation)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_rf_train, y_train)
rf_preds_val = rf_model.predict(X_rf_val)
evaluate_model("Random Forest (65% weight branch) - VALIDATION", y_val, rf_preds_val)

# Model 2: Linear Regression (Validation)
lr_model = LinearRegression()
lr_model.fit(X_lr_train, y_train)
lr_preds_val = lr_model.predict(X_lr_val)
evaluate_model("Linear Regression (35% weight branch) - VALIDATION", y_val, lr_preds_val)


### VALIDATION SET RESULTS ###

--- Random Forest (65% weight branch) - VALIDATION ---
R-squared: 0.8954
MAE: 16.7039
RMSE: 35.3264

--- Linear Regression (35% weight branch) - VALIDATION ---
R-squared: 0.9997
MAE: 0.7200
RMSE: 1.7991



In [ ]:
print("### TEST SET RESULTS ###\n")

# Model 1: Random Forest (Test)
rf_preds_test = rf_model.predict(X_rf_test)
evaluate_model("Random Forest (65% weight branch) - TEST", y_test, rf_preds_test)

# Model 2: Linear Regression (Test)
lr_preds_test = lr_model.predict(X_lr_test)
evaluate_model("Linear Regression (35% weight branch) - TEST", y_test, lr_preds_test)


### TEST SET RESULTS ###

--- Random Forest (65% weight branch) - TEST ---
R-squared: 0.8882
MAE: 17.7791
RMSE: 38.0687

--- Linear Regression (35% weight branch) - TEST ---
R-squared: 0.9998
MAE: 0.7690
RMSE: 1.4203



In [ ]:
# THE HYBRID MODEL YIELD CLASS
class HybridYieldModel:
    def __init__(self, rf, lr, enc, rf_f, lr_f):
        self.rf = rf
        self.lr = lr
        self.encoder = enc
        self.rf_f = rf_f
        self.lr_f = lr_f

    def predict(self, raw_input):
        if 'Plots' in raw_input.columns and ('Hectare' not in raw_input.columns or raw_input['Hectare'].isnull().any()):
            raw_input['Hectare'] = raw_input['Plots'] / 15.0

        cats = self.encoder.transform(raw_input[['State', 'District']])
        num_feats = raw_input[['Hectare', 'longitude', 'latitude']].values
        lr_input = np.hstack([cats, num_feats])
        
        pred_rf = self.rf.predict(raw_input[self.rf_f])[0]
        pred_lr = self.lr.predict(lr_input)[0]

        final_yield = (0.65 * pred_rf) + (0.35 * pred_lr)
        
        ph_val = raw_input['PH'].values[0]
        if ph_val < 5.5: status = "Warning: Soil too Acidic"
        elif ph_val > 7.5: status = "Warning: Soil too Alkaline"
        else: status = "Optimal"

        return {
            "Computed_Hectares": round(float(raw_input['Hectare'].values[0]), 2),
            "Estimated_Yield_MT": round(float(final_yield), 2),
            "Soil_Health_Status": status,
            "Weights": {"Climate_RF": "65%", "Regional_LR": "35%"}
        }

In [ ]:
# SAVED EVERYTHING TO ONE .PKL FILE
final_bundle = HybridYieldModel(rf_model, lr_model, encoder, rf_features, lr_features)
joblib.dump(final_bundle, 'val_hybrid_crop_model.pkl')

print("Process Complete! 'val_hybrid_crop_model.pkl' is ready for deployment.")

Process Complete! 'val_hybrid_crop_model.pkl' is ready for deployment.
